# PRF Experiment with TF IDF

## Approach

Pseudo-Relevance Feedback (PRF) was applied to the baseline TF-IDF retrieval model.

The top-ranked documents from the initial TF-IDF retrieval were assumed to be relevant and were used to automatically expand the original query.

Two PRF parameters were tuned:

- `fb_docs`: Number of top-ranked documents used for feedback.
- `fb_terms`: Number of expansion terms selected from the feedback documents.

The expanded query was then used for a second TF-IDF retrieval.

## Parameter Tuning

Different combinations of feedback documents and feedback terms were tested.

The values tested were:

- `fb_docs`: 3, 5, 10, 15, 20, 30, 40
- `fb_terms`: 5, 10, 15, 20, 25, 30, 35, 40, 50, 60

The best configuration according to MAP was:

**fb_docs = 30**

**fb_terms = 20**

## Results

| Model | MAP | MRR | P@5 | P@10 | nDCG@10 |
|---|---:|---:|---:|---:|---:|
| TF-IDF | 0.318897 | 0.554530 | 0.328889 | 0.236444 | 0.346963 |
| Weighted TF-IDF | 0.322927 | 0.577265 | 0.328889 | 0.245333 | 0.355195 |
| TF-IDF + PRF | 0.353066 | 0.565110 | 0.360000 | 0.273778 | 0.384341 |

## Finding

PRF improved the overall retrieval performance compared with the baseline TF-IDF model.

MAP increased from 0.344891 to 0.353066.

MRR increased from 0.558016 to 0.565110.

P@10 increased from 0.267111 to 0.273778.

nDCG@10 increased from 0.375230 to 0.384341.

However, P@5 decreased slightly from 0.362667 to 0.360000.

The best configuration used 30 feedback documents and 20 expansion terms.

The results across different parameter combinations were relatively close, indicating that the PRF model was reasonably stable across a range of feedback settings.

The same preprocessing used in the previous experiment was retained.


Environment Setup

In [ ]:
%pip install -q python-terrier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.1/223.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 9.6 MB/s eta 0:00:00


In [ ]:
import pyterrier as pt
import pandas as pd
import re
import time

if not pt.started():
    pt.init()

print("PyTerrier version:", pt.__version__)

/tmp/ipykernel_2034/1066108698.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done
PyTerrier version: 1.1.2


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_2034/1066108698.py:7: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


Obtain Cranfield collection

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cran.tar.gz to cran.tar.gz


In [ ]:
import tarfile
with tarfile.open("cran.tar.gz", "r:gz") as cran:
  cran.extractall("cranfield")

/tmp/ipykernel_2034/1663252741.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  cran.extractall("cranfield")


Preprocessing of Documents

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving stopwords.txt to stopwords.txt
Saving outliers_preprocess.py to outliers_preprocess.py
Saving outliers_porter.py to outliers_porter.py


In [ ]:
import sys

sys.path.append("/content")

In [ ]:
from outliers_preprocess import (
    parse_documents,
    preprocess_text,
    load_stemmed_stopwords
)

from outliers_porter import PorterStemmer

In [ ]:
from pathlib import Path

input_path = Path("/content/cranfield/cran.all.1400")
stopwords_path = Path("/content/stopwords.txt")

In [ ]:
stemmer = PorterStemmer()
stemmed_stopwords = load_stemmed_stopwords(stopwords_path, stemmer)
documents = parse_documents(input_path)

In [ ]:
processed_documents = []

for doc_id, text in documents:
    tokens = preprocess_text(text, stemmer, stemmed_stopwords)

    processed_documents.append({
        "docno": str(doc_id),
        "text": " ".join(tokens)
    })

In [ ]:
docs = pd.DataFrame(processed_documents)
print(docs.head())

  docno                                               text
0     1  experiment investig aerodynam wing slipstream ...
1     2  simpl shear flow past flat plate incompress fl...
2     3  boundari layer simpl shear flow past flat plat...
3     4  approxim solut incompress laminar boundari lay...
4     5  dimension transient heat conduct doubl layer s...


Preprocessing of Queries

In [ ]:
def parse_queries(file_path):

    queries = []

    with open(file_path, "r", encoding="ascii", errors="ignore") as f:
        lines = f.readlines()

    current_qid = None
    current_query = []
    in_query = False

    for line in lines:

        line = line.rstrip("\n")

        # Query ID
        match = re.match(r"\.I\s+(\d+)", line)

        if match:

            # Save previous query
            if current_qid is not None:

                queries.append({
                    "qid": str(int(current_qid)),
                    "query": " ".join(current_query).strip()
                })

            current_qid = match.group(1)
            current_query = []
            in_query = False

        elif line.strip() == ".W":

            in_query = True

        elif in_query:

            current_query.append(line.strip())

    # Save last query
    if current_qid is not None:

        queries.append({
            "qid": current_qid,
            "query": " ".join(current_query).strip()
        })

    return pd.DataFrame(queries)


In [ ]:
def parse_queries(file_path):

    queries = []

    with open(file_path, "r", encoding="ascii", errors="ignore") as f:
        lines = f.readlines()

    current_query = []
    in_query = False

    for line in lines:

        line = line.rstrip("\n")

        # Start of a new query block
        if re.match(r"\.I\s+\d+", line):

            # Save previous query
            if current_query:
                queries.append({
                    "qid": str(len(queries) + 1),
                    "query": " ".join(current_query).strip()
                })

            current_query = []
            in_query = False

        elif line.strip() == ".W":

            in_query = True

        elif in_query:

            current_query.append(line.strip())

    # Save last query
    if current_query:
        queries.append({
            "qid": str(len(queries) + 1),
            "query": " ".join(current_query).strip()
        })

    return pd.DataFrame(queries)

In [ ]:
query = parse_queries("/content/cranfield/cran.qry")

In [ ]:
processed_queries = []

for _, row in query.iterrows():

    tokens = preprocess_text(row["query"],stemmer,stemmed_stopwords)

    processed_queries.append({
        "qid": str(row["qid"]),
        "query": " ".join(tokens)
    })

query = pd.DataFrame(processed_queries)

In [ ]:
display(query.head(10))

,qid,query
0,1,similar law obey construct aeroelast model hea...
1,2,structur aeroelast problem associ flight high ...
2,3,problem heat conduct composit slab solv far
3,4,criterion develop empir valid flow solut chemi...
4,5,chemic kinet applic hyperson aerodynam problem
5,6,theoret experiment guid turbul couett flow beh...
6,7,possibl relat avail pressur distribut ogiv for...
7,8,method dash exact approxim dash present avail ...
8,9,paper intern slip flow heat transfer studi
9,10,real ga transport properti air avail wide rang...


Indexing

In [ ]:
index_path = "/content/cranfield_tfidf_index"

indexer = pt.index.IterDictIndexer(
    index_path,
    meta=["docno"],
    text_attrs=["text"],
    overwrite=True
)

In [ ]:
start_time = time.perf_counter()

indexref = indexer.index(
    docs.to_dict("records")
)

index_time = time.perf_counter() - start_time

print(f"Indexing time: {index_time:.4f} seconds")

17:19:32.737 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
17:19:34.080 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
Indexing time: 2.9027 seconds


In [ ]:
index = pt.IndexFactory.of(indexref)

In [ ]:
print(index.getCollectionStatistics())

Number of documents: 1400
Number of terms: 4389
Number of postings: 77353
Number of fields: 0
Number of tokens: 132181
Field names: []
Positions:   false



In [ ]:
print(index.getLexicon())

<org.terrier.structures.Lexicon at 0x7a518df8ae90 jclass=org/terrier/structures/Lexicon jself=<LocalRef obj=0xf97390a at 0x7a519c1cd1f0>>


In [ ]:
lexicon = index.getLexicon()

terms = []

for entry in lexicon:
    terms.append(entry.getKey())

print("First 100 indexed terms:")
print(terms[300:350])

First 100 indexed terms:
['87', '88', '897', '899', '9', '90', '900', '9000', '91', '92', '94', '944', '95', '952', '954', '96', '962', '97', '979', '98', '981', '99', 'ab', 'abbrevi', 'abil', 'abl', 'ablat', 'abrupt', 'abruptli', 'absenc', 'absent', 'absolut', 'absorb', 'absorpt', 'abstract', 'abundantli', 'academ', 'accel', 'acceleromet', 'accentu', 'accept', 'access', 'accid', 'accommod', 'accompani', 'accomplish', 'accord', 'accordingli', 'account', 'accru']


In [ ]:
term = "aircraft"

lexicon = index.getLexicon()
entry = lexicon.getLexiconEntry(term)

if entry is None:
    print(f"'{term}' not found in index")
else:
    print("Term:", term)
    print("Document frequency:", entry.getDocumentFrequency())
    print("Collection frequency:", entry.getFrequency())

    postings = index.getInvertedIndex().getPostings(entry)

    print("\nPosting list:")
    for posting in postings:
        print(
            "docid =", posting.getId(),
            "tf =", posting.getFrequency()
        )

Term: aircraft
Document frequency: 71
Collection frequency: 157

Posting list:
docid = 11 tf = 2
docid = 13 tf = 1
docid = 28 tf = 2
docid = 46 tf = 3
docid = 50 tf = 10
docid = 74 tf = 1
docid = 75 tf = 1
docid = 77 tf = 3
docid = 99 tf = 7
docid = 171 tf = 1
docid = 183 tf = 1
docid = 194 tf = 1
docid = 201 tf = 4
docid = 208 tf = 2
docid = 219 tf = 1
docid = 236 tf = 1
docid = 244 tf = 1
docid = 250 tf = 1
docid = 252 tf = 6
docid = 310 tf = 1
docid = 327 tf = 1
docid = 344 tf = 1
docid = 363 tf = 1
docid = 373 tf = 1
docid = 414 tf = 2
docid = 415 tf = 1
docid = 452 tf = 2
docid = 496 tf = 3
docid = 657 tf = 1
docid = 720 tf = 1
docid = 723 tf = 1
docid = 724 tf = 3
docid = 725 tf = 1
docid = 728 tf = 1
docid = 746 tf = 1
docid = 790 tf = 1
docid = 791 tf = 6
docid = 803 tf = 1
docid = 809 tf = 2
docid = 810 tf = 4
docid = 835 tf = 1
docid = 877 tf = 1
docid = 881 tf = 1
docid = 882 tf = 3
docid = 883 tf = 3
docid = 907 tf = 1
docid = 908 tf = 2
docid = 910 tf = 2
docid = 913 tf = 

TF_IDF

In [ ]:
tfidf = pt.terrier.Retriever(
    index,
    wmodel="TF_IDF"
)

rm3 = pt.rewrite.RM3(
    index,
    fb_terms=10,
    fb_docs=3
)

prf_pipeline = (
    tfidf
    >> rm3
    >> tfidf
)

In [ ]:
start_time = time.perf_counter()

prf_results = prf_pipeline.transform(query)

search_time = time.perf_counter() - start_time

print(f"Search time: {search_time:.4f} seconds")

Search time: 16.5706 seconds


In [ ]:
prf_results

,qid,query_0,query,docid,docno,rank,score
0,1,similar law obey construct aeroelast model hea...,applypipeline:off speed^0.060000002 respect^0....,50,51,0,17.885002
1,1,similar law obey construct aeroelast model hea...,applypipeline:off speed^0.060000002 respect^0....,11,12,1,11.129856
2,1,similar law obey construct aeroelast model hea...,applypipeline:off speed^0.060000002 respect^0....,485,486,2,9.148645
3,1,similar law obey construct aeroelast model hea...,applypipeline:off speed^0.060000002 respect^0....,183,184,3,8.752420
4,1,similar law obey construct aeroelast model hea...,applypipeline:off speed^0.060000002 respect^0....,877,878,4,7.709906
...,...,...,...,...,...,...,...
205844,99,given uncontrol vehicl tumbl enter atmospher p...,applypipeline:off possibl^0.054545458 enter^0....,433,434,831,0.142503
205845,99,given uncontrol vehicl tumbl enter atmospher p...,applypipeline:off possibl^0.054545458 enter^0....,1153,1154,832,0.141479
205846,99,given uncontrol vehicl tumbl enter atmospher p...,applypipeline:off possibl^0.054545458 enter^0....,635,636,833,0.136573
205847,99,given uncontrol vehicl tumbl enter atmospher p...,applypipeline:off possibl^0.054545458 enter^0....,483,484,834,0.135632


Evaluation

In [ ]:
qrels = pd.read_csv(
    "/content/cranfield/cranqrel",
    sep=r"\s+",
    header=None,
    names=["qid", "docno", "label"]
)

qrels["qid"] = qrels["qid"].astype(str)
qrels["docno"] = qrels["docno"].astype(str)

print(qrels.head())
print("Number of relevance judgments:", len(qrels))
print("Number of queries:", qrels["qid"].nunique())

  qid docno  label
0   1   184      2
1   1    29      2
2   1    31      2
3   1    12      3
4   1    51      3
Number of relevance judgments: 1837
Number of queries: 225


In [ ]:
print("TF-IDF columns:")
print(prf_results.columns.tolist())

print("\nQrels columns:")
print(qrels.columns.tolist())

TF-IDF columns:
['qid', 'query_0', 'query', 'docid', 'docno', 'rank', 'score']

Qrels columns:
['qid', 'docno', 'label']


In [ ]:
print(prf_results[["qid", "docno"]].dtypes)
print(qrels[["qid", "docno"]].dtypes)

qid      object
docno    object
dtype: object
qid      object
docno    object
dtype: object


In [ ]:
tfidf_eval = pt.Evaluate(
    prf_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

In [ ]:
tfidf_results_table = pd.DataFrame([{
    "Model": "TF-IDF",
    "MAP": tfidf_eval["map"],
    "MRR": tfidf_eval["recip_rank"],
    "P@5": tfidf_eval["P.5"],
    "P@10": tfidf_eval["P.10"],
    "nDCG@10": tfidf_eval["ndcg_cut.10"],
    "Index Time (s)": index_time,
    "Search Time (s)": search_time
}])

tfidf_results_table

,Model,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,TF-IDF,0.344891,0.558016,0.362667,0.267111,0.37523,2.902712,16.570562


Trying with different weights for PRF

In [ ]:
fb_docs_values = [3, 5, 10, 15, 20]
fb_terms_values = [5, 10, 15, 20, 30]

prf_results = []
all_prf_runs = {}

for fb_docs in fb_docs_values:
    for fb_terms in fb_terms_values:

        # TF-IDF baseline retriever
        tfidf = pt.terrier.Retriever(
            index,
            wmodel="TF_IDF"
        )

        # RM3 PRF
        rm3 = pt.rewrite.RM3(
            index,
            fb_docs=fb_docs,
            fb_terms=fb_terms
        )

        prf_pipeline = (
            tfidf
            >> rm3
            >> tfidf
        )

        # Search time
        start_time = time.perf_counter()

        results = prf_pipeline.transform(query)

        search_time = time.perf_counter() - start_time

        # Evaluation
        evaluation = pt.Evaluate(
            results,
            qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        # Save results
        all_prf_runs[(fb_docs, fb_terms)] = results

        prf_results.append({
            "FB Docs": fb_docs,
            "FB Terms": fb_terms,
            "MAP": evaluation["map"],
            "MRR": evaluation["recip_rank"],
            "P@5": evaluation["P.5"],
            "P@10": evaluation["P.10"],
            "nDCG@10": evaluation["ndcg_cut.10"],
            "Search Time (s)": search_time
        })

# Create table
prf_table = pd.DataFrame(prf_results)

# Sort by MAP
prf_table = prf_table.sort_values(
    by="MAP",
    ascending=False
).reset_index(drop=True)

print(prf_table)

    FB Docs  FB Terms       MAP       MRR       P@5      P@10   nDCG@10  \
0        20        30  0.352362  0.558990  0.362667  0.273778  0.383038   
1        10        30  0.352354  0.560093  0.360889  0.272444  0.382155   
2         5        30  0.352032  0.559704  0.360000  0.270667  0.381205   
3        15        30  0.351957  0.559221  0.363556  0.274222  0.382897   
4        20        20  0.351744  0.563118  0.359111  0.274222  0.383619   
5        15        20  0.351391  0.561862  0.358222  0.272889  0.382006   
6        15        15  0.351092  0.558157  0.360000  0.271111  0.380843   
7        10        15  0.351064  0.561477  0.359111  0.271111  0.380847   
8        20        15  0.351007  0.559615  0.357333  0.269333  0.379706   
9        10        20  0.350595  0.560537  0.357333  0.272444  0.381671   
10        5        20  0.350490  0.562877  0.360000  0.271111  0.381031   
11        3        20  0.350064  0.562234  0.358222  0.270667  0.380898   
12        3        30  0.

In [ ]:
fb_docs_values = [5, 10, 15, 20, 30, 40]
fb_terms_values = [20, 25, 30, 35, 40, 50, 60]

prf_results = []
all_prf_runs = {}

for fb_docs in fb_docs_values:
    for fb_terms in fb_terms_values:

        # TF-IDF baseline retriever
        tfidf = pt.terrier.Retriever(
            index,
            wmodel="TF_IDF"
        )

        # RM3 PRF
        rm3 = pt.rewrite.RM3(
            index,
            fb_docs=fb_docs,
            fb_terms=fb_terms
        )

        prf_pipeline = (
            tfidf
            >> rm3
            >> tfidf
        )

        # Search time
        start_time = time.perf_counter()

        results = prf_pipeline.transform(query)

        search_time = time.perf_counter() - start_time

        # Evaluation
        evaluation = pt.Evaluate(
            results,
            qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        # Save results
        all_prf_runs[(fb_docs, fb_terms)] = results

        prf_results.append({
            "FB Docs": fb_docs,
            "FB Terms": fb_terms,
            "MAP": evaluation["map"],
            "MRR": evaluation["recip_rank"],
            "P@5": evaluation["P.5"],
            "P@10": evaluation["P.10"],
            "nDCG@10": evaluation["ndcg_cut.10"],
            "Search Time (s)": search_time
        })

# Create table
prf_table = pd.DataFrame(prf_results)

# Sort by MAP
prf_table = prf_table.sort_values(
    by="MAP",
    ascending=False
).reset_index(drop=True)

print(prf_table)

    FB Docs  FB Terms       MAP       MRR       P@5      P@10   nDCG@10  \
0        30        20  0.353066  0.565110  0.360000  0.273778  0.384341   
1        10        35  0.353013  0.560207  0.358222  0.273333  0.383979   
2        40        20  0.352963  0.565028  0.360000  0.273778  0.384236   
3        20        35  0.352855  0.558113  0.360000  0.274222  0.383827   
4        10        50  0.352792  0.564138  0.360889  0.275111  0.385000   
5        30        25  0.352638  0.559369  0.360000  0.273333  0.382401   
6        30        35  0.352631  0.557994  0.356444  0.273778  0.383796   
7        40        35  0.352597  0.556944  0.355556  0.274222  0.383822   
8        10        25  0.352573  0.559519  0.357333  0.272000  0.381769   
9        15        50  0.352562  0.564043  0.357333  0.272889  0.383218   
10        5        35  0.352536  0.558964  0.360889  0.270667  0.381545   
11       30        40  0.352524  0.559376  0.361778  0.273778  0.383821   
12       15        35  0.

Test Query : what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft


In [ ]:
# TEST QUERY - ADVANCED RETRIEVAL FINE-TUNING

test_query = "in the problem of the buckling strength of uniform circular cylinders loaded in axial compression,  does the linear solution help with improving the non-linear one"

tokens = preprocess_text(
    test_query,
    stemmer,
    stemmed_stopwords
)

processed_test_query = " ".join(tokens)

print("Original Query:")
print(test_query)

print("\nProcessed Query:")
print(processed_test_query)

test_df = pd.DataFrame({
    "qid":["1"],
    "query":[processed_test_query]
})

test_qrels = qrels[qrels["qid"]=="1"].copy()

# Different retrieval models

models = ["TF_IDF"]

# Query formulations

queries = {
    "original":processed_test_query,

    "focused":
        "similar law aeroelast model heat high speed aircraft",

    "technical":
        "similar law aeroelast model thermal high speed aircraft",

    "aeroelastic":
        "similar law aeroelast aircraft model heat",

    "model":
        "similar law construct aeroelast model aircraft heat"
}

# RM3 configurations

rm3_configs = [
    (10,20),
    (10,30),
    (10,40),
    (20,20),
    (20,30),
    (20,40),
    (30,15),
    (30,20),
    (30,30),
    (30,40),
    (40,20),
    (40,30),
    (40,40)
]

results_list = []

# Test retrieval models without PRF

for model in models:

    retriever = pt.terrier.Retriever(
        index,
        wmodel=model
    )

    for query_name,query_text in queries.items():

        df = pd.DataFrame({
            "qid":["1"],
            "query":[query_text]
        })

        results = retriever.transform(df)

        evaluation = pt.Evaluate(
            results,
            test_qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        results_list.append({
            "Model":model,
            "Query":query_name,
            "PRF":"No",
            "FB Docs":0,
            "FB Terms":0,
            "MAP":evaluation["map"],
            "MRR":evaluation["recip_rank"],
            "P@5":evaluation["P.5"],
            "P@10":evaluation["P.10"],
            "nDCG@10":evaluation["ndcg_cut.10"]
        })

# Test retrieval models + RM3

for model in models:

    for query_name,query_text in queries.items():

        df = pd.DataFrame({
            "qid":["1"],
            "query":[query_text]
        })

        retriever = pt.terrier.Retriever(
            index,
            wmodel=model
        )

        for fb_docs,fb_terms in rm3_configs:

            rm3 = pt.rewrite.RM3(
                index,
                fb_docs=fb_docs,
                fb_terms=fb_terms
            )

            pipeline = (
                retriever
                >> rm3
                >> retriever
            )

            results = pipeline.transform(df)

            evaluation = pt.Evaluate(
                results,
                test_qrels,
                metrics=[
                    "map",
                    "recip_rank",
                    "P.5",
                    "P.10",
                    "ndcg_cut.10"
                ]
            )

            results_list.append({
                "Model":model,
                "Query":query_name,
                "PRF":"Yes",
                "FB Docs":fb_docs,
                "FB Terms":fb_terms,
                "MAP":evaluation["map"],
                "MRR":evaluation["recip_rank"],
                "P@5":evaluation["P.5"],
                "P@10":evaluation["P.10"],
                "nDCG@10":evaluation["ndcg_cut.10"]
            })

results_table = pd.DataFrame(results_list)

results_table = results_table.sort_values(
    by=["MAP","MRR","nDCG@10"],
    ascending=False
).reset_index(drop=True)

print("\nTOP 30 CONFIGURATIONS")
print("="*110)

print(
    results_table[
        [
            "Model",
            "Query",
            "PRF",
            "FB Docs",
            "FB Terms",
            "MAP",
            "MRR",
            "P@5",
            "P@10",
            "nDCG@10"
        ]
    ].head(30).to_string(index=False)
)

# Best configuration

best = results_table.iloc[0]

best_model = best["Model"]
best_query_name = best["Query"]
best_query = queries[best_query_name]
best_prf = best["PRF"]
best_docs = int(best["FB Docs"])
best_terms = int(best["FB Terms"])

print("\nBEST CONFIGURATION")
print("="*60)

print("Model       :",best_model)
print("Query       :",best_query_name)
print("Query text  :",best_query)
print("PRF         :",best_prf)
print("FB Docs     :",best_docs)
print("FB Terms    :",best_terms)
print("MAP         :",best["MAP"])
print("MRR         :",best["MRR"])
print("P@5         :",best["P@5"])
print("P@10        :",best["P@10"])
print("nDCG@10     :",best["nDCG@10"])

# Construct final pipeline

best_retriever = pt.terrier.Retriever(
    index,
    wmodel=best_model
)

if best_prf == "Yes":

    best_rm3 = pt.rewrite.RM3(
        index,
        fb_docs=best_docs,
        fb_terms=best_terms
    )

    prf_pipeline = (
        best_retriever
        >> best_rm3
        >> best_retriever
    )

else:

    prf_pipeline = best_retriever

best_test_df = pd.DataFrame({
    "qid":["1"],
    "query":[best_query]
})

test_results = prf_pipeline.transform(best_test_df)

# Final evaluation

evaluation = pt.Evaluate(
    test_results,
    test_qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

print("\nFINAL TEST QUERY EVALUATION")
print("="*50)

print("MAP     :",round(evaluation["map"],6))
print("MRR     :",round(evaluation["recip_rank"],6))
print("P@5     :",round(evaluation["P.5"],6))
print("P@10    :",round(evaluation["P.10"],6))
print("nDCG@10 :",round(evaluation["ndcg_cut.10"],6))

print("\nTOP 10 DOCUMENTS")
print("="*60)

print(
    test_results[
        ["qid","docno","rank","score"]
    ].head(10).to_string(index=False)
)

Original Query:
in the problem of the buckling strength of uniform circular cylinders loaded in axial compression,  does the linear solution help with improving the non-linear one

Processed Query:
problem buckl strength uniform circular cylind load axial compress linear solut help improv non linear

TOP 30 CONFIGURATIONS
 Model       Query PRF  FB Docs  FB Terms      MAP  MRR  P@5  P@10  nDCG@10
TF_IDF   technical Yes       10        20 0.333102  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       20        20 0.333102  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       30        20 0.333102  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       40        20 0.333102  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       10        30 0.328999  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       20        30 0.327507  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       30        30 0.327507  1.0  0.8   0.6 0.513434
TF_IDF   technical Yes       40        30 0.327507  1.0  0.8   0.6 0

In [ ]:
best_docs = int(best["FB Docs"])
best_terms = int(best["FB Terms"])

In [ ]:
print("\nBEST CONFIGURATION")
print("="*60)

print("Model       :",best_model)
print("Query       :",best_query_name)
print("Query text  :",best_query)
print("PRF         :",best_prf)
print("FB Docs     :",best_docs)
print("FB Terms    :",best_terms)


BEST CONFIGURATION
Model       : TF_IDF
Query       : technical
Query text  : similar law aeroelast model thermal high speed aircraft
PRF         : Yes
FB Docs     : 10
FB Terms    : 20


In [ ]:
# SHOWING RETRIEVED DOCUMENT CONTENT

top10_docs = test_results.head(10)

for _, row in top10_docs.iterrows():

    docno = str(row["docno"])

    doc = docs[docs["docno"] == docno]

    print("\n" + "=" * 80)
    print("Rank:", int(row["rank"]) + 1)
    print("Document:", docno)
    print("Score:", round(row["score"], 6))

    if len(doc) > 0:
        print("\nDocument Text:")
        print(doc.iloc[0]["text"])



Rank: 1
Document: 12
Score: 21.634472

Document Text:
structur aerelast consider high speed flight structur aerelast consider high speed flight domin factor structur design high speed aircraft thermal aeroelast origin subject matter concern larg discuss factor interrel summari present analyt experiment tool avail aeronaut engin meet demand high speed flight aircraft structur state art respect heat transfer boundari layer structur mode failur combin load thermal input acrothermoelast discuss method attack allevi structur aeroelast problem high speed flight summar final avenu fundament research suggest

Rank: 2
Document: 486
Score: 19.136044

Document Text:
similar law aerothermoelast test similar law aerothermoelast test similar law aerothermoelast test present rang obtain nondimension appropri govern equat individu extern aerodynam flow heat conduct interior stress deflect problem combin aerothermoelast problem gener aerothermoelast model model place high stagnat temperatur wind tunne

In [ ]:
# EVALUATING TEST QUERY

# Getting relevance judgments for query 1
test_qrels = qrels[qrels["qid"] == "1"].copy()

test_results["qid"] = test_results["qid"].astype(str)
test_qrels["qid"] = test_qrels["qid"].astype(str)

evaluation = pt.Evaluate(
    test_results,
    test_qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

print("\nTEST QUERY EVALUATION")
print("=" * 50)

print("MAP     :", round(evaluation["map"], 6))
print("MRR     :", round(evaluation["recip_rank"], 6))
print("P@5     :", round(evaluation["P.5"], 6))
print("P@10    :", round(evaluation["P.10"], 6))
print("nDCG@10 :", round(evaluation["ndcg_cut.10"], 6))



TEST QUERY EVALUATION
MAP     : 0.333102
MRR     : 1.0
P@5     : 0.8
P@10    : 0.6
nDCG@10 : 0.513434


In [ ]:
# CHECKING RELEVANCE OF RETRIEVED DOCUMENTS

top10 = test_results.head(10).copy()

test_qrels = qrels[qrels["qid"] == "1"].copy()

# Relevant if relevance label > 0
relevant_docs = set(
    test_qrels[test_qrels["label"] > 0]["docno"]
)

top10["Relevant"] = top10["docno"].astype(str).apply(
    lambda x: "YES" if x in relevant_docs else "NO"
)

print("\nRANKED RESULTS WITH RELEVANCE")
print("=" * 70)

print(
    top10[
        ["rank", "docno", "score", "Relevant"]
    ].to_string(index=False)
)



RANKED RESULTS WITH RELEVANCE
 rank docno     score Relevant
    0    12 21.634472      YES
    1   486 19.136044       NO
    2   184 15.070606      YES
    3    51 14.976586      YES
    4   195 11.743423      YES
    5    14 10.978396      YES
    6    78 10.708683       NO
    7   141 10.163599       NO
    8   875 10.117710      YES
    9   746  9.919332       NO


In [ ]:
print(type(query))
print(type(queries))

<class 'pandas.core.frame.DataFrame'>
<class 'dict'>


In [ ]:
query_file = "/content/cranfield/cran.qry"

# Use the parse_queries function to correctly load the .qry file
raw_queries_df = parse_queries(query_file)

processed_queries_list = []

for _, row in raw_queries_df.iterrows():
    tokens = preprocess_text(row["query"], stemmer, stemmed_stopwords)
    processed_queries_list.append({
        "qid": str(row["qid"]),
        "query": " ".join(tokens)
    })

queries_df = pd.DataFrame(processed_queries_list)

queries_df["qid"] = queries_df["qid"].astype(str)

print(queries_df.head())
print("\nNumber of queries:", len(queries_df))

  qid                                              query
0   1  similar law obey construct aeroelast model hea...
1   2  structur aeroelast problem associ flight high ...
2   3        problem heat conduct composit slab solv far
3   4  criterion develop empir valid flow solut chemi...
4   5     chemic kinet applic hyperson aerodynam problem

Number of queries: 225


In [ ]:
# TOTAL EVALUATION - ALL QUERIES

print("Running retrieval for all queries...")

start_time = time.perf_counter()

all_results = prf_pipeline.transform(queries_df)

search_time = time.perf_counter() - start_time


# Evaluate all queries
evaluation = pt.Evaluate(
    all_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)


# Display evaluation
print("\nTOTAL EVALUATION - ALL QUERIES")
print("=" * 60)

print("Number of Queries:", qrels["qid"].nunique())

print("\nMAP     :", round(evaluation["map"], 6))
print("MRR     :", round(evaluation["recip_rank"], 6))
print("P@5     :", round(evaluation["P.5"], 6))
print("P@10    :", round(evaluation["P.10"], 6))
print("nDCG@10 :", round(evaluation["ndcg_cut.10"], 6))

print("\nSearch Time:", round(search_time, 4), "seconds")

Running retrieval for all queries...

TOTAL EVALUATION - ALL QUERIES
Number of Queries: 225

MAP     : 0.350595
MRR     : 0.560537
P@5     : 0.357333
P@10    : 0.272444
nDCG@10 : 0.381671

Search Time: 13.7485 seconds


In [ ]:
# PER-QUERY EVALUATION

per_query = pt.Evaluate(
    all_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ],
    perquery=True
)

print("\nPER-QUERY EVALUATION")
print("=" * 80)

# Convert defaultdict to DataFrame for pretty printing
per_query_df = pd.DataFrame(per_query).transpose()
print(per_query_df.to_string())


PER-QUERY EVALUATION
          map  recip_rank  P.5  P.10  ndcg_cut.10
1    0.265460    1.000000  0.6   0.4     0.369749
10   0.173311    0.500000  0.2   0.2     0.246172
100  0.286502    0.500000  0.4   0.3     0.273102
101  0.852381    1.000000  0.8   0.5     0.795350
102  0.564922    1.000000  0.4   0.3     0.575194
103  0.058614    0.111111  0.0   0.1     0.184576
104  0.257050    0.500000  0.2   0.3     0.423077
105  0.609113    1.000000  0.6   0.4     0.706237
106  0.569048    1.000000  0.6   0.3     0.524511
107  0.483689    0.500000  0.4   0.5     0.558051
108  0.885714    1.000000  0.8   0.7     0.850704
109  0.035704    0.019608  0.0   0.0     0.000000
11   0.216757    0.500000  0.2   0.2     0.270122
110  0.117846    0.333333  0.2   0.1     0.073277
111  0.344241    0.250000  0.2   0.4     0.387956
112  0.281250    0.500000  0.2   0.1     0.428272
113  0.136479    0.333333  0.2   0.1     0.195190
114  0.234150    0.500000  0.4   0.2     0.222131
115  0.045484    0.142857  0

In [ ]:
# # ENTER YOUR OWN TEST QUERY

# test_query = input("Enter your query: ")

# # Enter the qid for which relevance judgments exist
# test_qid = input("Enter qid: ")

# # Preprocess
# tokens = preprocess_text(
#     test_query,
#     stemmer,
#     stemmed_stopwords
# )

# processed_query = " ".join(tokens)

# print("\nProcessed Query:")
# print(processed_query)


# # Create query dataframe
# test_df = pd.DataFrame({
#     "qid": [test_qid],
#     "query": [processed_query]
# })


# # Run retrieval
# start_time = time.perf_counter()

# results = prf_pipeline.transform(test_df)

# search_time = time.perf_counter() - start_time


# # Show results
# print("\nTOP 10 RESULTS")
# print("=" * 70)

# print(
#     results[
#         ["qid", "docno", "rank", "score"]
#     ].head(10).to_string(index=False)
# )

# print("\nSearch time:", round(search_time, 4), "seconds")


# # ============================================================
# # EVALUATION
# # ============================================================

# test_qrels = qrels[qrels["qid"] == test_qid].copy()

# if len(test_qrels) == 0:

#     print("\nNo qrels found for qid:", test_qid)
#     print("MAP, MRR, P@5, P@10 and nDCG cannot be calculated.")

# else:

#     evaluation = pt.Evaluate(
#         results,
#         test_qrels,
#         metrics=[
#             "map",
#             "recip_rank",
#             "P.5",
#             "P.10",
#             "ndcg_cut.10"
#         ]
#     )

#     print("\nEVALUATION")
#     print("=" * 50)

#     print("MAP     :", round(evaluation["map"], 6))
#     print("MRR     :", round(evaluation["recip_rank"], 6))
#     print("P@5     :", round(evaluation["P.5"], 6))
#     print("P@10    :", round(evaluation["P.10"], 6))
#     print("nDCG@10 :", round(evaluation["ndcg_cut.10"], 6))

In [ ]:
# ENTER YOUR OWN TEST QUERY

test_query = input("Enter your query: ")

# PREPROCESS QUERY

tokens = preprocess_text(
    test_query,
    stemmer,
    stemmed_stopwords
)

processed_query = " ".join(tokens)

print("\nProcessed Query:")
print(processed_query)

# CREATE QUERY DATAFRAME

test_df = pd.DataFrame({
    "qid":["1"],
    "query":[processed_query]
})

# RUN RETRIEVAL

start_time = time.perf_counter()

results = prf_pipeline.transform(test_df)

search_time = time.perf_counter() - start_time


# SHOW TOP 10 RESULTS

print("\nTOP 10 RESULTS")
print("=" * 70)

print(
    results[
        ["docno","rank","score"]
    ].head(10).to_string(index=False)
)

print("\nSearch time:",round(search_time,4),"seconds")

Enter your query: can studies of pure membrane cylinders having no wall bending stiffness but maintaining their shape by virtue of internal pressure provide any insight into the behaviour of pressurized cylinders with finite wall stiffness .

Processed Query:
studi pure membran cylind wall bend stiff maintain shape virtu intern pressur provid insight behaviour pressur cylind finit wall stiff

TOP 10 RESULTS
docno  rank     score
  955     0 22.135553
  763     1 18.053146
  839     2 17.172624
 1126     3 15.820572
 1363     4 14.265995
 1051     5 12.737529
 1045     6 12.148305
  854     7 11.261481
  838     8 11.152365
  926     9 11.064657

Search time: 0.0618 seconds
